# Phase 2 - Testing

Workflow for testing all modules for phase 2 (agentic workflow). Execute top to bottom

### Query Rewriting
Takes care of formulating proper query for search in vector DB - that includes semantic query formatted properly for this purpose, along with standard keyword filters (price, other features)

Test simple query rewriting - creates optimal semantic query for vector search, along with filter hints for standard fields (price, features)

In [1]:
from chatshop.infra.llm_client import llm_client_from_settings
from chatshop.rag.query_rewriter import QueryRewriter

llm_client = llm_client_from_settings()

query_rewriter = QueryRewriter(llm_client)

In [2]:
query_rewriter.rewrite("Need headphones for commuting, under $100")

RewrittenQuery(semantic_query='noise-cancelling headphones portable long battery life', filter_hints={'max_price': 100, 'min_price': 0, 'min_rating': None, 'extra_filters': {'anc': True, 'use_cases': 'travel'}}, intent_summary='The user is looking for affordable noise-cancelling headphones suitable for commuting.')

Test a more complex query rewriting, where critical information about headphones is split among multiple messages

In [ ]:
from chatshop.rag.query_rewriter import _SYSTEM_PROMPT

messages = [
    {"role": "system", "content": _SYSTEM_PROMPT},
    {"role": "user", "content": "Hey, I need some new headphones."},
    {"role": "assistant", "content": "Sure — what will you mainly use them for?"},
    {"role": "user", "content": "Mostly when I go running."},
    {"role": "assistant", "content": "Got it. Do you have any budget in mind?"},
    {"role": "user", "content": "Yeah, ideally under 120."},
    {"role": "assistant", "content": "Do you prefer earbuds or over-ear?"},
]

query_rewriter.rewrite("Definitely earbuds. And I hate cables.", history=messages)

RewrittenQuery(semantic_query='wireless sport earbuds sweat-resistant stable fit', filter_hints={'max_price': 120.0, 'min_price': 0.0, 'min_rating': None, 'extra_filters': {'wireless': True, 'type': 'in-ear', 'use_cases': 'sport'}}, intent_summary='User is looking for wireless sport earbuds for running under $120.')

### Agent workflow - the Planner

The planner is the 'brain', responsible for judging what is next required step. Search for products? Respond to user? Ask for clarifications?

In [2]:
from chatshop.agent.planner import Planner        

planner = Planner(llm_client, query_rewriter)

Case 1 - a simple definite prompt

In [3]:
result_searchAction = planner.plan("I need headphones for commuting, under $100")
print(result_searchAction)

SearchAction(action='search', search_plan=SearchPlan(semantic_query='noise-cancelling headphones portable long battery life', filters=SearchFilters(max_price=100, min_price=0, min_rating=None, extra_filters={'anc': True, 'use_cases': 'travel'}), sort_by=None), reasoning_trace="The user's intent is clear: they are looking for headphones specifically for commuting and have a budget of under $100. This allows for a reasonable search query to be constructed based on these attributes.")


Case 2 - generic prompt

In [11]:
result_clarifyAction = planner.plan("Hi, what's up?")
print(result_clarifyAction)

ClarifyAction(action='clarify', question='What specific information or assistance do you need regarding headphones?', reasoning_trace="The user's intent is vague and does not provide any context about headphones or related queries. I need to ask a focused question to understand what they are looking for.")


Case 3 - vague prompt

In [8]:
result = planner.plan("Hi, I need god damn heaphones, the best there is")
print(result)

ClarifyAction(action='clarify', question='What specific features or use cases are you looking for in the best headphones?', reasoning_trace="The user's request for 'the best headphones' is vague as it does not specify any particular use case, preferences, or features they are looking for. To provide a meaningful recommendation, I need to clarify what they mean by 'best' — whether they are looking for sound quality, noise cancellation, comfort, or something else.")


In [9]:
long_conversation = [
    {"role": "user", "content": "Hey, I need some new headphones."},
    {"role": "assistant", "content": "Sure — what will you mainly use them for?"},
    {"role": "user", "content": "Mostly when I go running."},
    {"role": "assistant", "content": "Got it. Do you have any budget in mind?"},
    {"role": "user", "content": "Yeah, ideally under 120."},
    {"role": "assistant", "content": "Do you prefer earbuds or over-ear?"},
    {"role": "user", "content": "Definitely earbuds. And I hate cables."},
]

result = planner.plan(long_conversation)
print(result)

SearchAction(action='search', search_plan=SearchPlan(semantic_query='wireless sport earbuds sweat-resistant stable fit', filters=SearchFilters(max_price=120.0, min_price=0.0, min_rating=None, extra_filters={'wireless': True, 'type': 'in-ear', 'use_cases': 'sport'}), sort_by=None), reasoning_trace="The user's intent is clear: they want wireless earbuds for running, with a budget under $120. This allows for a reasonable search to find suitable products that meet these criteria.")


### Hybrid Search


In [4]:
from re import search

from chatshop.rag.retriever import Retriever
from chatshop.rag.hybrid_search import HybridSearch

retriever = Retriever()
hybrid_search = HybridSearch(retriever)



In [5]:
result_searchAction.search_plan

SearchPlan(semantic_query='noise-cancelling headphones portable long battery life', filters=SearchFilters(max_price=100, min_price=0, min_rating=None, extra_filters={'anc': True, 'use_cases': 'travel'}), sort_by=None)

In [6]:
search_results = hybrid_search.search(result_searchAction.search_plan)
for p in search_results.products:
    print(p.to_context_text())
    print("---")

Audio-Technica Nova 3
Audio-Technica QuietPoint ATH-ANC900BT brings Japanese acoustic precision and reliability to a feature-rich wireless headphone at an accessible price. The 40mm driver system is engineered for the natural, musical sound profile Audio-Technica has perfected over decades in professional studio monitor design. Hybrid Digital Active Noise Cancelling with four microphones produces effective isolation from transportation and environmental noise during commutes. The folding design with premium carrying 
Price: $99.00 | Brand: Audio-Technica | Type: over-ear | Wireless: Yes | ANC: Yes | Battery: 50h | Driver: 40.0mm | Use cases: travel, office
---
Anker Soundcore Life Q35
Anker Soundcore Life Q35 delivers premium features at an unbeatable price. LDAC codec support ensures you're receiving Hi-Res wireless audio with three times more data than standard Bluetooth connections. Three levels of adaptive noise cancellation intelligently analyze your environment and filter distrac